<a href="https://colab.research.google.com/github/ArjunHirani/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

## 1. My lane as an ML task (type)


**Task type: a classification model used in service of a ranking/scoring output.**

My lane (freestyle — Growth/Recovery/Momentum Prediction) sounds like a classification question on the surface: will this page's clicks decline over the next 30 days (yes/no)? But the actual deployed artifact is a **ranked list**, because a content reviewer only has capacity to act on a limited number of pages per week (see Week 1 framing: ~50/week against thousands of candidates). The task-type mapping in the framing skill puts "which ones first?" under Ranking/Scoring with precision@K as the metric, and "will this one decline?" under Classification with ROC-AUC. My case is genuinely both: I will train a binary classifier (decline / not-decline), but I will *evaluate and deploy it as a ranking* — using `predict_proba` as the score, not the thresholded label. This distinction matters because a model can have mediocre overall classification accuracy and still be an excellent ranker for a reviewer who only ever looks at the top 20-50, which is precisely the deployment reality here.

In [1]:
# Making the task-type reasoning concrete: which row of the framing skill's mapping
# table applies, and why this is BOTH classification and ranking, not one or the other.
task_type_mapping = {
    "question_shape": "Which ones first? (ranking/scoring)  +  Will this one decline? (classification)",
    "model_trained_as": "binary classifier -> decline_next30 (yes/no)",
    "model_deployed_as": "ranking -- sort by predict_proba(decline), reviewer works top-K down",
    "why_both": "reviewer capacity is the real constraint (~50 pages/week), not overall accuracy"
}
for k, v in task_type_mapping.items():
    print(f"{k:20s}: {v}")

question_shape      : Which ones first? (ranking/scoring)  +  Will this one decline? (classification)
model_trained_as    : binary classifier -> decline_next30 (yes/no)
model_deployed_as   : ranking -- sort by predict_proba(decline), reviewer works top-K down
why_both            : reviewer capacity is the real constraint (~50 pages/week), not overall accuracy


## 2. Target or proxy


**Target: `target_decline_next30`** — an **observed outcome**, not a rule-defined label.

I deliberately do NOT reuse the starter notebooks' `is_declining_label` (`trend_direction == "down"`), because that column is itself *defined* from `trend_pct` — exactly the label-trap the data skill and Notebook 02's leakage lesson warn about. Instead, I build my target from the starter CSV's own paired 30-day windows, which already exist in the schema: `clicks_prev_30d` (the window before) and `clicks_last_30d` (the window after). `target_decline_next30 = 1` if `clicks_last_30d < clicks_prev_30d`, else `0`. This is a genuinely **observed, measured outcome** across two real time windows — the same structure I'll later need to build properly from the warehouse's `report_date` column for the real capstone, just at a smaller scale here.

I also restrict to pages with real prior activity (`clicks_prev_30d >= 5`) — without this filter, over half the rows have zero clicks in both windows, which would make "decline" indistinguishable from "no data," a different kind of proxy-label mistake.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Why the activity filter matters: without it, "decline" is confounded with "no data at all"
zero_both = ((df["clicks_prev_30d"] == 0) & (df["clicks_last_30d"] == 0)).mean()
print(f"Share of ALL pages with zero clicks in BOTH windows (would corrupt a naive target): {zero_both:.3f}")

active = df[df["clicks_prev_30d"] >= 5].copy()
active["target_decline_next30"] = (active["clicks_last_30d"] < active["clicks_prev_30d"]).astype(int)

print(f"\nActive pages (clicks_prev_30d >= 5): {len(active)} of {len(df)} ({len(active)/len(df):.1%})")
print(f"Target distribution: {active['target_decline_next30'].value_counts().to_dict()}")
print(f"Share declining next-30d among active pages: {active['target_decline_next30'].mean():.3f}")

Share of ALL pages with zero clicks in BOTH windows (would corrupt a naive target): 0.523

Active pages (clicks_prev_30d >= 5): 5019 of 30000 (16.7%)
Target distribution: {1: 3161, 0: 1858}
Share declining next-30d among active pages: 0.630


## 3. Success metric


**Metric: Precision@50** — of the top 50 pages my model ranks as highest decline-risk, what fraction actually declined?

I defend this over plain accuracy or ROC-AUC because of the deployment reality named in Section 1: a reviewer works down a ranked list with fixed weekly capacity, so what matters is whether the *top* of the list is trustworthy — not whether the model is well-calibrated across the whole population. "Good" means: **meaningfully above the base rate**, the same honest framing the starter pipeline itself uses (baseline rule Precision@50 = 0.240 vs. random forest 0.740 on the starter's own label). For my target, the base rate among active pages is the number to beat, computed below.

In [3]:
base_rate = active["target_decline_next30"].mean()
print(f"Base rate to beat (Precision@50 of a RANDOM top-50 selection, in expectation): {base_rate:.3f}")
print("Success bar for this task: Precision@50 of a trained, properly-validated model")
print("must beat this base rate by a clear, honest margin -- measured on a client-holdout")
print("split, not in-sample (per the Notebook 02 lesson).")

Base rate to beat (Precision@50 of a RANDOM top-50 selection, in expectation): 0.630
Success bar for this task: Precision@50 of a trained, properly-validated model
must beat this base rate by a clear, honest margin -- measured on a client-holdout
split, not in-sample (per the Notebook 02 lesson).


## 4. The unit of analysis, as a real dataframe


**One row = one active content page, observed over a prior 30-day window, with an outcome measured in the following 30-day window** (within this single starter snapshot). This is *not* just "one row = one page" — it's specifically a windowed observation. In the eventual full-warehouse version of this lane, the same unit of analysis generalizes to one row = one **(client, content item, snapshot month)** panel observation, since the warehouse's `fact_content_daily_performance` table is genuinely time-indexed rather than a single snapshot.

In [4]:
# The real dataframe: one row = one active page's prior-window slice + next-window outcome
unit_of_analysis_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "word_count",
    "clicks_last_30d", "target_decline_next30"
]
print(active[unit_of_analysis_cols].head(8).to_string(index=False))
print(f"\nShape: {active.shape[0]} rows x {len(unit_of_analysis_cols)} shown columns")

          content_id         client_id    content_type   main_intent  impressions_prev_30d  clicks_prev_30d  sessions_prev_30d  content_age_days  word_count  clicks_last_30d  target_decline_next30
content_304f48230142 client_f369cb89fc keyword article transactional                   987               13                  9               187      3221.0                2                      1
content_331d6c4de07b client_19581e27de keyword article    commercial                  4206               17                 26               463         NaN               22                      0
content_5e6c160719bc client_6208ef0f77 keyword article informational                 13828                8                 14                90      3807.0                9                      0
content_d8ee6cc6d642 client_19581e27de keyword article    commercial                  6441              117                136               329         NaN              112                      1
content_42fb2ca

## 5. Why ML beats a fixed rule here


I tested this directly rather than asserting it, and found something more interesting than "combining signals helps a little":

- Using only **genuinely clean, non-overlapping prior-window signals** (the `_prev_30d` columns and static content properties), individual correlations with the target are all weak (max ~0.12), and **neither a single-signal rule nor a simple 3-signal hand-combination beats the base rate** — Precision@50 came out to 0.540, actually *below* the 0.630 base rate. A hand-written rule here isn't just suboptimal, it's actively worse than doing nothing.
- Meanwhile, `scroll_rate` looked deceptively strong on its own (Precision@50 = 0.860) — but `scroll_rate` is a **90-day trailing aggregate that overlaps the exact 30-day outcome window** I'm predicting. That's a subtler version of the `trend_direction`/`trend_pct` leakage trap from Notebook 02: a signal that looks powerful only because it partially *contains* the answer, not because it predicts it.

Together, this is real evidence — not an assumption — for why this problem needs a properly trained, honestly validated model rather than an if-statement: the clean signal is genuinely weak and hard to combine correctly by hand, and the tempting-looking strong signal is a leakage risk in disguise. Telling these apart is exactly the kind of work a fixed rule can't do safely, but a model built with careful feature-window discipline and held-out validation can.

In [5]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = active["target_decline_next30"]
y_arr = y.values

# Clean, non-overlapping prior-window signals only
active["prior_window_ctr"] = (active["clicks_prev_30d"] / active["impressions_prev_30d"].replace(0, np.nan)).fillna(0)

clean_signals = ["impressions_prev_30d", "sessions_prev_30d", "prior_window_ctr", "content_age_days", "word_count"]
print("Correlation of CLEAN (non-overlapping) prior-window signals with target:")
for col in clean_signals:
    s = active[col].replace([np.inf, -np.inf], np.nan).fillna(active[col].median())
    print(f"  {col:22s}: {s.corr(y):+.3f}")

combo_clean = (
    (1 - active["prior_window_ctr"].rank(pct=True)) +
    (1 - active["impressions_prev_30d"].rank(pct=True)) +
    active["content_age_days"].fillna(0).rank(pct=True)
).values
p50_combo_clean = precision_at_k(combo_clean, y_arr, 50)
base_rate = y.mean()
print(f"\nPrecision@50, 3-signal CLEAN hand-combined rule: {p50_combo_clean:.3f}  (base rate: {base_rate:.3f})")

# The deceptive, leakage-risk signal
scroll_score = active["scroll_rate"].fillna(0).values
p50_scroll = precision_at_k(scroll_score, y_arr, 50)
scroll_corr = active["scroll_rate"].fillna(0).corr(y)
print(f"\n[CAUTION] Precision@50 using scroll_rate alone: {p50_scroll:.3f}  (correlation: {scroll_corr:+.3f})")
print("scroll_rate is a 90-day trailing aggregate that OVERLAPS the last_30d outcome window --")
print("this apparent strength is a same-window leakage risk, not a trustworthy prior signal.")

Correlation of CLEAN (non-overlapping) prior-window signals with target:
  impressions_prev_30d  : +0.016
  sessions_prev_30d     : +0.050
  prior_window_ctr      : +0.124
  content_age_days      : -0.033
  word_count            : -0.034

Precision@50, 3-signal CLEAN hand-combined rule: 0.540  (base rate: 0.630)

[CAUTION] Precision@50 using scroll_rate alone: 0.860  (correlation: +0.134)
scroll_rate is a 90-day trailing aggregate that OVERLAPS the last_30d outcome window --
this apparent strength is a same-window leakage risk, not a trustworthy prior signal.
